# Download/Pre-process CFS forecast data
Lindsay Fitzpatrick
ljob@umich.edu
**Initial Version:** 08/28/2024  
**Last Updated:** 08/18/2025 

This script automates the download and preprocessing of **Climate Forecast System (CFS)** forecast data for the Great Lakes region. The data is sourced in GRIB2 format from either **AWS** or **NCEI** repositories.

### Key Functionality
- Downloads raw CFS forecast files for specified dates and forecast hours.
- Processes the GRIB2 files to calculate key atmospheric metrics:
  - Total precipitation
  - Evaporation
  - 2-meter air temperature (averaged separately over lake and land surfaces)
- Applies a geographic mask to isolate values over lake and land areas using `GL_mask.nc`.
- Saves the processed results into a local **SQLite database** (`cfs_forecast_data.db`), either by appending to an existing file or creating a new one.

### Required Input Files
- **`GL_mask.nc`**  
  A NetCDF file containing geographic masks to distinguish lake and land grid cells in the Great Lakes region.

- **`cfs_forecast_data.db`** *(optional)*  
  If provided, the script will append new forecast data to this database; otherwise, it will create a new database with the appropriate schema.

This script forms the first step in the CNBS forecasting pipeline, ensuring that clean, lake-specific forecast variables are prepared for use in downstream modeling and prediction.

In [1]:
import os
import sys
import pandas as pd
import netCDF4 as nc

In [2]:
# Add the path to the src directory (two levels up)
sys.path.append(os.path.abspath('../../'))
from src.database_utils import *
from src.data_processing import create_directory, process_grib_files
from src.hydro_utils import calculate_grid_cell_areas

## User Inputs
### Configuration: File Paths and Processing Options

This section defines key file paths and user-configurable options for downloading and processing CFS forecast data:

- **`local_path`**: Base directory where the repository is cloned and data folders are located.  
- **`download_dir`**: Directory where raw CFS GRIB2 files will be downloaded and temporarily stored.  
- **`input_dir`**: Directory containing required input files such as masks and scalers.  
- **`mask_file`**: Path to the Great Lakes mask file (`GL_mask.nc`) used for separating lake and land data.  
- **`database`**: Path to the SQLite database where processed CFS forecast data is stored.  

#### Data Source and Processing Control  
- **`source`**: Specifies the data source, either `'aws'` or `'ncei'`.  
- **`download_cfs`**: Toggle to enable or disable downloading new CFS data (`'yes'` or `'no'`).  
- **`process_cfs`**: Toggle to enable or disable processing of downloaded CFS data (`'yes'` or `'no'`).  
- **`delete_files`**: Option to delete raw GRIB2 files after processing to save storage (`'yes'` or `'no'`).  

#### Date Range Configuration  
- **`auto`**: When set to `'yes'`, automatically updates the database by detecting the last processed date and downloading data up to yesterday’s date.  
- If **`auto`** is `'no'`, specify manual start and end dates for data download and processing:  
  - **`start_date`**: Starting date for data retrieval (format: MM-DD-YYYY).  
  - **`end_date`**: Ending date for data retrieval (format: MM-DD-YYYY).  

These settings allow flexible control over data acquisition and processing, supporting both automated updates and manual testing or reprocessing.

In [3]:
# Directory where the repository is cloned
local_path = '/Users/ljob/Desktop/'

# Path to data directory
input_dir = local_path + 'cnbs-predictor/data/'

# Path the GL mask file
mask_file = input_dir + 'input/GL_mask.nc'

# Path to save downloaded data
download_dir = local_path + 'Data/CFS/'

# Path to the CFS forecast data database
database = local_path + 'Data/cfs_forecast_data.db'

# Data source: specify either 'aws' or 'ncei'
source = 'aws'

# Do you need to download CFS data? ('yes' or 'no')
download_cfs = 'yes'

# Do you want to process the CFS data? ('yes' or 'no')
process_cfs = 'yes'

# Should grib files be deleted after processing? ('yes' or 'no')
delete_files = 'no'

# Auto mode will automatically open the existing database, pull the last entered date to determine the start date, 
# and set the end date to yesterday, making the database 'up-to-date'. If 'no', you can manually enter a start and 
# end date (ideal for testing or if you need to redownload/reprocess specific time frames).
auto = 'yes'

# Specify the start and end dates if auto mode above is set to 'no' (Format: MM-DD-YYYY)
start_date = '08-01-2024'
end_date = '08-01-2025'

### Preset Variables

This section defines key preset variables used throughout the CFS data download and processing workflow:

- **`products`**: List of CFS forecast product types to retrieve.  
  - `'pgb'`: Pressure-level forecast data  
  - `'flx'`: Surface flux data

- **`utc`**: List of UTC initialization hours for which forecasts will be downloaded (`00`, `06`, `12`, `18`).

- **`mask_variables`**: Names of the lake and land regions used for applying the spatial mask.  
  Each entry corresponds to a specific Great Lake (e.g., Erie, Ontario, Michigan-Huron, Superior), with distinctions for lake surface and surrounding land areas.

- **`bucket_name`**: AWS S3 bucket name (`noaa-cfs-pds`) used to access the public NOAA CFS forecast data.

These presets standardize the data sources, time intervals, and spatial definitions used in the forecast extraction and preprocessing steps.

In [4]:
## Presets ##
products = ['pgb','flx']
utc = ['00','06','12','18']

# Define mask variables
mask_variables = ['eri_lake','eri_land',
                  'ont_lake','ont_land',
                  'mih_lake','mih_land',
                  'sup_lake','sup_land']

#AWS bucket name to locate the CFS forecast
bucket_name = 'noaa-cfs-pds'

## Begin Script

### Initialize Output Directory and Database

This section ensures the necessary file structure and database are in place before downloading or processing any data:

- **`create_directory(download_dir)`**:  
  Checks if the specified download directory exists.  
  - If it exists, a message is printed: *"Directory already exists."*  
  - If not, the directory is created and a confirmation message is printed: *"Directory created."*

- **`open_cfs_db(database)`**:  
  Opens the existing SQLite database (`cfs_forecast_data.db`) if it exists.  
  If it does not, a new database is created with the appropriate schema for storing processed CFS forecast data.

This setup step ensures that all required storage locations are ready for the script to execute without errors.

In [5]:
create_directory(download_dir)
open_cfs_db(database)

Directory '/Users/ljob/Desktop/Data/CFS/' already exists.


(<sqlite3.Connection at 0x104dba6b0>, <sqlite3.Cursor at 0x10f8a59c0>)

### Determine Date Range for Forecast Download and Processing

This section defines the date range over which the script will operate, based on the user's selected mode:

- When **`auto = 'yes'`**:  
  The script automatically checks the `cfs_forecast_data.db` database for the most recent recorded CFS run.  
  - The **start date** is set to the day after the last recorded run.  
  - The **end date** is set to **yesterday**, ensuring the database stays up to date with the latest available forecasts.

- When **`auto = 'no'`**:  
  The user manually specifies both the start and end dates.  
  This is especially useful for testing or initializing a new database with historical forecasts.

Before proceeding, the script prints a message confirming the selected date range, for example:  
`Starting from: 05-22-2025 00Z and continuing through: 06-10-2025 18Z`

This ensures transparency and gives the user a final opportunity to confirm the configured date range before data download and processing begins.

In [6]:
if auto == 'yes':
        # Fetch next cfs_run date and use yesterday's date for the end date
        start_date_i = get_next_cfs_run(database, 'cfs_forecast_data')
        end_date_i = (datetime.now() - timedelta(days=1)).strftime("%m-%d-%Y") + " 18"
        # Validate dates
        if start_date_i >= end_date_i:
            print("The csv files are up-to-date.")
        else:
            print(f"Starting from: {start_date_i}Z and continuing through: {end_date_i}Z")

else:
    # Ensure both start_date and end_date have hour info
    start_date = (start_date + " 00") if len(start_date) == 10 else start_date
    end_date = (end_date + " 18") if len(end_date) == 10 else end_date

    # Convert to datetime objects for comparison
    start_date_i = datetime.strptime(start_date, "%m-%d-%Y %H")
    end_date_i = datetime.strptime(end_date, "%m-%d-%Y %H")

    # Validate dates
    if start_date_i == end_date_i:
        print(start_date_i)
        print("The csv files are up-to-date.")
    elif start_date_i > end_date_i:
        print(start_date_i)
        print("There is an error in the input dates. Please try again.")
    else:
        print(f"Starting from: {start_date_i.strftime('%m-%d-%Y %H')}Z and continuing through: {end_date_i.strftime('%m-%d-%Y %H')}Z")

date_array = pd.date_range(start=start_date_i, end=end_date_i, freq='6h')

Starting from: 08-11-2025 00Z and continuing through: 08-18-2025 18Z


### Load Mask File and Calculate Grid Cell Areas

This section loads the spatial mask and prepares geographic data used to extract and process the Great Lakes region from global CFS datasets:

- **`mask_ds = nc.Dataset(mask_file)`**  
  Opens the NetCDF mask file (`GL_mask.nc`), which defines lake and land grid cells for the Great Lakes.

- **`mask_lat` / `mask_lon`**  
  Extracts the latitude and longitude arrays from the mask file. These coordinates define the spatial extent of the region of interest and are used to crop the global forecast data down to the Great Lakes domain.

- **`calculate_grid_cell_areas(mask_lon, mask_lat)`**  
  Computes the surface area of each grid cell based on the geographic coordinates.  
  These areas are later used for spatial averaging and calculating total quantities (e.g., precipitation volume).

This spatial setup ensures accurate and efficient processing of CFS data specific to the Great Lakes basin.

In [7]:
# Open the mask file and calculate the grid cell areas
mask_ds = nc.Dataset(mask_file)
mask_lat = mask_ds.variables['latitude'][:]
mask_lon = mask_ds.variables['longitude'][:]
area = calculate_grid_cell_areas(mask_lon, mask_lat)

### Loop Through Forecast Dates: Download, Process, and Store Data

This section loops through each date in the user-defined `date_array` to download, process, and store CFS forecast data for the Great Lakes region.

For each date:
1. **Creates a daily subdirectory** within the `download_dir` to store GRIB2 files specific to that date.
2. **Downloads GRIB2 files** based on the selected data source:
   - If **`source = 'aws'`**, constructs the appropriate S3 path and uses `download_grb2_aws()` to fetch files.
   - If **`source = 'ncei'`**, builds a public NCEI URL and checks for availability before downloading using `download_grb2_ncei()`.
   - Validates the source input and skips the date if files are not available.
3. If **`process_cfs = 'yes'`**, the script:
   - Processes the downloaded GRIB2 files using `process_grib_files()`, which calculates atmospheric metrics (e.g., precipitation, evaporation, temperature) and applies spatial masks.
   - Saves the results to the specified SQLite database (`cfs_forecast_data.db`) under the `cfs_forecast_data` table.
4. If **`delete_files = 'yes'`**, the script deletes the local GRIB2 directory to conserve storage space.

Each date’s progress is printed to the console (e.g., `"Beginning Files for 2025-06-10 00:00:00"`), and a final message confirms when all dates have been processed:  
**`"Process Complete"`**

This loop forms the core of the data pipeline, ensuring all available CFS forecast data is up-to-date, processed, and stored efficiently.

In [8]:
for date in date_array:
    print(f"Beginning Files for {date}.")

    date_str = date.strftime('%Y%m%d%H')
    YYYY, MM, DD, HH = date.strftime('%Y'), date.strftime('%m'), date.strftime('%d'), date.strftime('%H')

    download_path = f'{download_dir}{YYYY}{MM}{DD}/'
    os.makedirs(download_path, exist_ok=True)

    # ===== Download GRIB2 Files =====
    if download_cfs.lower() == 'yes':
        for product in products:
            if source == 'aws':
                url_path = f'cfs.{YYYY}{MM}{DD}/{HH}/monthly_grib_01/'
                download_grb2_aws(product, bucket_name, url_path, download_path)
            elif source == 'ncei':
                base_url = 'https://www.ncei.noaa.gov/data/climate-forecast-system/access/operational-9-month-forecast/monthly-means/'
                url_path = f'{base_url}/{YYYY}/{YYYY}{MM}/{YYYY}{MM}{DD}/{YYYY}{MM}{DD}{HH}/'
                if not url_path or not check_url_exists(url_path):
                    print(f"No files available for {date}. Skipping.")
                else:
                    download_grb2_ncei(product, url_path, download_path)
            
            else:
                print('Input source does not exist. Source must be aws or ncei.')
    
    # ===== Process GRIB Files =====
    if process_cfs.lower() == 'yes':
        process_grib_files(
            download_path, database, 'cfs_forecast_data', date_str,
            mask_lat, mask_lon, mask_ds, mask_variables, area
        )

        if delete_files.lower() == 'yes':
            try:
                os.rmdir(download_path)
            except OSError:
                print(f"Cannot delete directory {download_path}. Skipping.")
    
    print(f'Done with {date}.')
print("Process Complete")

Beginning Files for 2025-08-11 00:00:00.
Downloaded: cfs.20250811/00/monthly_grib_01/pgbf.01.2025081100.202508.avrg.grib.grb2
Downloaded: cfs.20250811/00/monthly_grib_01/pgbf.01.2025081100.202509.avrg.grib.grb2
Downloaded: cfs.20250811/00/monthly_grib_01/pgbf.01.2025081100.202510.avrg.grib.grb2
Downloaded: cfs.20250811/00/monthly_grib_01/pgbf.01.2025081100.202511.avrg.grib.grb2
Downloaded: cfs.20250811/00/monthly_grib_01/pgbf.01.2025081100.202512.avrg.grib.grb2
Downloaded: cfs.20250811/00/monthly_grib_01/pgbf.01.2025081100.202601.avrg.grib.grb2
Downloaded: cfs.20250811/00/monthly_grib_01/pgbf.01.2025081100.202602.avrg.grib.grb2
Downloaded: cfs.20250811/00/monthly_grib_01/pgbf.01.2025081100.202603.avrg.grib.grb2
Downloaded: cfs.20250811/00/monthly_grib_01/pgbf.01.2025081100.202604.avrg.grib.grb2
Downloaded: cfs.20250811/00/monthly_grib_01/pgbf.01.2025081100.202605.avrg.grib.grb2
Downloaded: cfs.20250811/00/monthly_grib_01/flxf.01.2025081100.202508.avrg.grib.grb2
Downloaded: cfs.20250811

### Close Mask File

After all required data (latitude, longitude, mask variables, and grid cell areas) have been extracted and used, the script explicitly closes the `mask_ds` NetCDF file using:

```python
mask_ds.close()

In [9]:
mask_ds.close()